# Image segmentation with U-Net: worked solution

Worked solution for the exercise in the lesson **[Image segmentation with U-Net](https://ramkumarmd.com/medical-ai/deep-learning-imaging/segmentation-unet)** (*Deep learning & imaging* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](03-segmentation-unet.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Segmentation assigns a label to every pixel, outlining a lesion, organ or wound rather than just saying whether it is present. The U-Net architecture does this with an encoder that captures context, a decoder that restores detail, and skip connections that carry fine detail across. Results are usually scored with the Dice coefficient, which measures overlap between the predicted and true outlines. Report the spread of scores, not only the average, because a model with a good mean Dice can still fail badly on some images.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A tiny U-Net in PyTorch

Synthetic scans with masks, a tiny U-Net, the Dice score and training. About ten seconds on a CPU.

In [ ]:
import numpy as np
import torch
from torch import nn

def make_scans(n, seed, size=32):
    """Synthetic scans with one or two irregular 'lesions', and the true pixel mask for each."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:size, :size]
    images, masks = rng.normal(0, 1, (n, size, size)), np.zeros((n, size, size))
    for i in range(n):
        for _ in range(rng.integers(1, 3)):
            cy, cx = rng.integers(6, size - 6, 2)
            ry, rx = rng.uniform(2, 5, 2)
            masks[i] = np.maximum(masks[i], ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 < 1)
        images[i] += 1.5 * masks[i]
    to_t = lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(1)
    return to_t(images), to_t(masks)

def block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU())

class TinyUNet(nn.Module):
    """Encoder-decoder with a skip connection: the U-Net idea at the smallest useful size."""
    def __init__(self):
        super().__init__()
        self.enc1, self.enc2 = block(1, 8), block(8, 16)
        self.pool, self.up = nn.MaxPool2d(2), nn.ConvTranspose2d(16, 8, 2, stride=2)
        self.dec1 = block(16, 8)                       # 8 upsampled + 8 from the skip connection
        self.out = nn.Conv2d(8, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)                              # full resolution: detail
        e2 = self.enc2(self.pool(e1))                  # half resolution: context
        d1 = self.dec1(torch.cat([self.up(e2), e1], dim=1))
        return self.out(d1)                            # one logit per pixel

def dice(pred_mask, true_mask, eps=1e-6):
    inter = (pred_mask * true_mask).sum(dim=(1, 2, 3))
    return ((2 * inter + eps) / (pred_mask.sum(dim=(1, 2, 3)) + true_mask.sum(dim=(1, 2, 3)) + eps))

X_train, M_train = make_scans(1500, seed=1)
X_test, M_test = make_scans(300, seed=2)
torch.manual_seed(0)
unet = TinyUNet()
opt = torch.optim.Adam(unet.parameters(), lr=2e-3)
for epoch in range(1, 9):
    for idx in torch.randperm(len(X_train)).split(32):
        opt.zero_grad()
        nn.functional.binary_cross_entropy_with_logits(unet(X_train[idx]), M_train[idx]).backward()
        opt.step()
    if epoch in (1, 4, 8):
        with torch.no_grad():
            d = dice((unet(X_test) > 0).float(), M_test)
        print(f"epoch {epoch}: mean Dice on test scans {d.mean():.3f}")

# epoch 1: mean Dice on test scans 0.000
# epoch 4: mean Dice on test scans 0.864
# epoch 8: mean Dice on test scans 0.891

### Comparing with a baseline, and looking at the worst cases

Continuing from the previous block: a brightness-threshold baseline and the worst 5% of U-Net scores.

In [ ]:
with torch.no_grad():
    d = dice((unet(X_test) > 0).float(), M_test)
    threshold_only = dice((X_test > 0.75).float(), M_test)       # naive: bright pixels = lesion
print(f"Simple brightness threshold: mean Dice {threshold_only.mean():.3f}")
print(f"U-Net: mean Dice {d.mean():.3f}; worst 5% of scans below {torch.quantile(d, 0.05):.3f}")

# Simple brightness threshold: mean Dice 0.270
# U-Net: mean Dice 0.891; worst 5% of scans below 0.769

## Exercise

**Goal:** Train and evaluate a U-Net, improve it for small lesions, and report results responsibly.

**Data:** Synthetic scans and masks in the notebook; optional open segmentation dataset under its licence (Generated in the notebook)

### Step 1

Reproduce the tiny U-Net and plot predicted masks for the five worst-scoring test scans.

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    logits = unet(X_test)
    d = dice((logits > 0).float(), M_test)
worst = torch.argsort(d)[:5]
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for col, i in enumerate(worst):
    for row, (img, title) in enumerate([(X_test[i, 0], "scan"), (M_test[i, 0], "true mask"),
                                        ((logits[i, 0] > 0).float(), f"predicted, Dice {d[i]:.2f}")]):
        axes[row, col].imshow(img, cmap="gray"); axes[row, col].set_title(title, fontsize=8); axes[row, col].axis("off")
plt.tight_layout(); plt.show()
print("Worst five Dice scores:", [round(float(d[i]), 3) for i in worst])

The worst five include a complete miss (Dice 0.0) and several partial ones. Look at the figure to see what they have
in common; small or faint lesions, where a few wrong pixels change Dice a lot, are a common cause. Looking at the
failures is how you learn what a mean of 0.89 hides.

### Step 2

Replace the loss with a combination of cross-entropy and Dice loss and compare early epochs and final Dice.

In [ ]:
def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    return (1 - (2 * inter + eps) / (p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps)).mean()

def train_unet(loss_name, X=X_train, M=M_train, model_cls=TinyUNet, epochs=8, report_epochs=(1, 4, 8)):
    torch.manual_seed(0)
    net = model_cls()
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    history = {}
    for epoch in range(1, epochs + 1):
        for idx in torch.randperm(len(X)).split(32):
            opt.zero_grad()
            out = net(X[idx])
            bce = nn.functional.binary_cross_entropy_with_logits(out, M[idx])
            loss = bce if loss_name == "BCE" else bce + dice_loss(out, M[idx])
            loss.backward(); opt.step()
        if epoch in report_epochs:
            with torch.no_grad():
                history[epoch] = dice((net(X_test) > 0).float(), M_test).mean().item()
    return net, history

for name in ["BCE", "BCE + Dice"]:
    _, hist = train_unet(name)
    print(f"{name:<11}", "  ".join(f"epoch {e}: {v:.3f}" for e, v in hist.items()))

Lesion pixels are a small fraction of each scan, so plain cross-entropy is dominated by the background and starts by
predicting "no lesion" everywhere (Dice 0 after the first epoch). Adding a Dice term rewards overlap directly. Here
both losses are still at 0 after one epoch; the combined loss is slightly ahead at epoch 4 (0.876 against 0.864) and
at the end (0.895 against 0.891). The benefit of a Dice term is usually larger when lesions are rarer than this.

### Step 3

Make some lesions much smaller and report per-lesion detection rate alongside mean Dice.

In [ ]:
def make_small_lesion_scans(n, seed, size=32):
    """As make_scans, but each scan has one normal lesion and one very small one (radius 1-1.6 pixels)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:size, :size]
    images, masks, small = rng.normal(0, 1, (n, size, size)), np.zeros((n, size, size)), np.zeros((n, size, size))
    for i in range(n):
        for radius_range, store in [((2, 5), None), ((1.0, 1.6), small)]:
            cy, cx = rng.integers(6, size - 6, 2)
            ry, rx = rng.uniform(*radius_range, 2)
            blob = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 < 1
            masks[i] = np.maximum(masks[i], blob)
            if store is not None:
                store[i] = blob
        images[i] += 1.5 * masks[i]
    to_t = lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(1)
    return to_t(images), to_t(masks), to_t(small)

Xs_test, Ms_test, small_test = make_small_lesion_scans(300, seed=7)
with torch.no_grad():
    pred = (unet(Xs_test) > 0).float()
mean_dice = dice(pred, Ms_test).mean().item()
detected = ((pred * small_test).sum(dim=(1, 2, 3)) > 0).float().mean().item()   # any overlap counts as detected
print(f"Mean Dice with small lesions present: {mean_dice:.3f}")
print(f"Small lesions detected (any overlap): {detected:.1%}")

Mean Dice falls only modestly (from 0.891 to 0.826), because the small lesions contribute few pixels to each scan's
overlap, while the per-lesion detection rate shows that most of them (about 69%) are missed. In screening for small lesions (for example
microaneurysms in diabetic retinopathy), detection rate is the clinically important number, so report both.

### Step 4

Add a second downsampling level and compare results and training time.

In [ ]:
import time

class TwoLevelUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = block(1, 8), block(8, 16), block(16, 32)
        self.pool = nn.MaxPool2d(2)
        self.up2, self.dec2 = nn.ConvTranspose2d(32, 16, 2, stride=2), block(32, 16)
        self.up1, self.dec1 = nn.ConvTranspose2d(16, 8, 2, stride=2), block(16, 8)
        self.out = nn.Conv2d(8, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(e3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

for name, cls in [("one downsampling level", TinyUNet), ("two downsampling levels", TwoLevelUNet)]:
    start = time.perf_counter()
    net, hist = train_unet("BCE + Dice", model_cls=cls, report_epochs=(8,))
    secs = time.perf_counter() - start
    params = sum(p.numel() for p in net.parameters())
    print(f"{name:<24} final Dice {hist[8]:.3f}  parameters {params:>6,}  training time {secs:.0f} s")

The second level sees a wider context (each output pixel depends on a larger part of the image) at the cost of
more parameters and time. On 32x32 scans with small lesions there is little extra context to use, so gains are
modest; on large images with big structures, depth matters more. Timing depends on your hardware.

### Step 5

Write a results paragraph that reports the mean, the spread and the failure cases.

In [ ]:
final_dice = d
print(f"n={len(final_dice)}, mean {final_dice.mean():.3f}, median {final_dice.median():.3f}, "
      f"IQR {torch.quantile(final_dice, 0.25):.3f}-{torch.quantile(final_dice, 0.75):.3f}, "
      f"min {final_dice.min():.3f}, scans below 0.7: {(final_dice < 0.7).sum().item()}")

Example results paragraph (fill in the numbers printed above):

"On 300 held-out synthetic scans, the U-Net achieved a mean Dice coefficient of 0.89 (median 0.91, interquartile
range 0.87-0.93), compared with 0.27 for a brightness-threshold baseline. Performance was not uniform: four scans
scored below 0.7, including one complete miss (examples in the figure). When a second, very small lesion was added
to each scan, only 31% of those small lesions were detected, although mean Dice fell only to 0.83. Results are on synthetic data from the same generator as the training set, so they
indicate whether the method works, not how it would perform on real images."

---

Lesson: [https://ramkumarmd.com/medical-ai/deep-learning-imaging/segmentation-unet](https://ramkumarmd.com/medical-ai/deep-learning-imaging/segmentation-unet). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.